In [1]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import dowhy
from dowhy import CausalModel
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

c:\Users\Gurveer\anaconda3\envs\ds-portfolio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
# Cell 2 — Generate Realistic Flight Delay Dataset
np.random.seed(42)
n = 8000

# Causal variables — these CAUSE delays
weather_severity  = np.random.beta(2, 5, n)        # 0=clear 1=severe
nas_congestion    = np.random.beta(3, 4, n)         # NAS = National Airspace System
carrier_issues    = np.random.beta(2, 6, n)         # airline-specific problems
late_aircraft     = np.random.choice([0,1], n, p=[0.75, 0.25])

# Confounders
season = np.random.choice(['winter','spring','summer','fall'], n,
                           p=[0.30, 0.22, 0.28, 0.20])
season_factor = {'winter': 1.4, 'spring': 0.9, 'summer': 1.1, 'fall': 0.8}
s_factor = np.array([season_factor[s] for s in season])

airport = np.random.choice(['ORD','ATL','JFK','LAX','DFW','BOS','SEA'], n)
airport_factor = {'ORD':1.3,'ATL':1.1,'JFK':1.2,'LAX':0.9,'DFW':1.0,'BOS':1.1,'SEA':0.95}
a_factor = np.array([airport_factor[a] for a in airport])

airline = np.random.choice(['Delta','United','American','Southwest','JetBlue'], n)
airline_factor = {'Delta':0.85,'United':1.1,'American':1.05,'Southwest':0.95,'JetBlue':0.9}
al_factor = np.array([airline_factor[a] for a in airline])

# Structural causal model — delay is a function of causes
delay_minutes = (
    60  * weather_severity  * s_factor +
    45  * nas_congestion    * a_factor +
    30  * carrier_issues    * al_factor +
    25  * late_aircraft +
    np.random.normal(0, 8, n)
).clip(0)

is_delayed = (delay_minutes > 15).astype(int)

df = pd.DataFrame({
    'weather_severity': weather_severity.round(4),
    'nas_congestion':   nas_congestion.round(4),
    'carrier_issues':   carrier_issues.round(4),
    'late_aircraft':    late_aircraft,
    'season':           season,
    'airport':          airport,
    'airline':          airline,
    'delay_minutes':    delay_minutes.round(1),
    'is_delayed':       is_delayed
})

print(f"Dataset shape:   {df.shape}")
print(f"Delay rate:      {df['is_delayed'].mean()*100:.1f}%")
print(f"Avg delay:       {df['delay_minutes'].mean():.1f} minutes")
print(f"\nDelay by season:")
print(df.groupby('season')['delay_minutes'].mean().round(1).sort_values(ascending=False))
print(f"\nDelay by airline:")
print(df.groupby('airline')['delay_minutes'].mean().round(1).sort_values(ascending=False))

Dataset shape:   (8000, 9)
Delay rate:      98.7%
Avg delay:       53.2 minutes

Delay by season:
season
winter    58.2
summer    53.3
spring    50.2
fall      48.7
Name: delay_minutes, dtype: float64

Delay by airline:
airline
American     54.2
United       54.0
JetBlue      53.4
Southwest    52.7
Delta        51.8
Name: delay_minutes, dtype: float64


In [3]:
# Cell 3 — Predictive Model (GBM)
# Encode categoricals
df_encoded = pd.get_dummies(df, columns=['season', 'airport', 'airline'], drop_first=True)

features = [c for c in df_encoded.columns 
            if c not in ['delay_minutes', 'is_delayed']]

X = df_encoded[features]
y = df_encoded['is_delayed']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

gbm = GradientBoostingClassifier(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, random_state=42
)
gbm.fit(X_train_scaled, y_train)
gbm_proba = gbm.predict_proba(X_test_scaled)[:, 1]

print(f"GBM ROC-AUC: {roc_auc_score(y_test, gbm_proba):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, gbm.predict(X_test_scaled)))

# Feature importance
importance_df = pd.DataFrame({
    'feature':    features,
    'importance': gbm.feature_importances_
}).sort_values('importance', ascending=False).head(10)

fig = px.bar(importance_df, x='importance', y='feature',
             orientation='h',
             title='Top 10 Feature Importances — GBM Delay Predictor',
             template='plotly_dark')
fig.update_layout(width=800, height=450, yaxis=dict(autorange='reversed'))
fig.show()

GBM ROC-AUC: 0.9499

Classification Report:
              precision    recall  f1-score   support

           0       0.50      0.19      0.28        21
           1       0.99      1.00      0.99      1579

    accuracy                           0.99      1600
   macro avg       0.74      0.59      0.63      1600
weighted avg       0.98      0.99      0.98      1600



In [8]:
# Cell 4 — Causal Inference via Propensity Score Matching
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# Binarize treatment: high weather severity vs low
causal_df = df[['weather_severity', 'nas_congestion',
                'carrier_issues', 'late_aircraft',
                'is_delayed']].copy()

causal_df['treatment'] = (causal_df['weather_severity'] > 
                           causal_df['weather_severity'].median()).astype(int)

# Confounders to control for
confounders = ['nas_congestion', 'carrier_issues', 'late_aircraft']

# Step 1: Estimate propensity scores
ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(causal_df[confounders], causal_df['treatment'])
causal_df['propensity_score'] = ps_model.predict_proba(
    causal_df[confounders])[:, 1]

print("=== Propensity Score Summary ===")
print(causal_df.groupby('treatment')['propensity_score'].describe().round(4))

=== Propensity Score Summary ===
            count    mean     std     min     25%     50%     75%     max
treatment                                                                
0          4000.0  0.4995  0.0164  0.4370  0.4885  0.5005  0.5113  0.5452
1          4000.0  0.5006  0.0163  0.4387  0.4905  0.5018  0.5120  0.5415


In [9]:
# Cell 5 — Causal Effect Estimation with DoWhy
identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)
print("=== Identified Estimand ===")
print(identified_estimand)

estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.linear_regression",
    control_value=0,
    treatment_value=1
)

print("\n=== Estimate ===")
print(estimate)

val = estimate.value
print(f"\nCausal estimate:  {val:.4f}")
print(f"\nInterpretation: A unit increase in weather severity")
print(f"causes a {val:.4f} increase in delay probability")
print(f"holding nas_congestion, carrier_issues, and late_aircraft constant.")

weather_effect_value = val

=== Identified Estimand ===
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
         d                        
───────────────────(E[is_delayed])
d[weather_severity]               
Estimand assumption 1, Unconfoundedness: If U→{weather_severity} and U→is_delayed then P(is_delayed|weather_severity,,U) = P(is_delayed|weather_severity,)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
         d                        
───────────────────(E[is_delayed])
d[weather_severity]               
Estimand assumption 1, Unconfoundedness: If U→{weather_severity} and U→is_delayed then P(is_delayed|weather_severity,,U) = P(is_delayed|weather_severity,)


=== Estimate ===
*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estima

In [10]:
# Cell 6 — Refutation Tests & Causal Comparison
import os

# Placebo refutation
refute_placebo = model.refute_estimate(
    identified_estimand, estimate,
    method_name="placebo_treatment_refuter",
    placebo_type="permute",
    num_simulations=50
)
print("=== Placebo Refutation Test ===")
print(refute_placebo)

# Causal effect comparison across all causes
causes = ['weather_severity', 'nas_congestion', 'carrier_issues', 'late_aircraft']
effects = []

print("\n=== Causal Effect Per Factor ===")
for cause in causes:
    m = CausalModel(
        data=causal_df,
        treatment=cause,
        outcome='is_delayed',
        graph=causal_graph
    )
    est  = m.identify_effect(proceed_when_unidentifiable=True)
    eff  = m.estimate_effect(
        est,
        method_name="backdoor.linear_regression",
        control_value=0,
        treatment_value=1
    )
    effects.append({'cause': cause, 'causal_effect': round(float(eff.value), 4)})
    print(f"  {cause:<25} causal effect = {eff.value:.4f}")

effects_df = pd.DataFrame(effects).sort_values('causal_effect', ascending=False)

fig = px.bar(effects_df, x='cause', y='causal_effect',
             color='causal_effect',
             color_continuous_scale='Reds',
             title='Causal Effect of Each Factor on Flight Delay Probability',
             template='plotly_dark')
fig.update_layout(width=750, height=430)
fig.show()

# Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-05-flight-delay\outputs'
os.makedirs(output_dir, exist_ok=True)
effects_df.to_csv(f'{output_dir}\\causal_effects.csv', index=False)
importance_df.to_csv(f'{output_dir}\\feature_importance.csv', index=False)
df.to_csv(f'{output_dir}\\flight_delay_dataset.csv', index=False)
print(f"\nExports saved to outputs/")

=== Placebo Refutation Test ===
Refute: Use a Placebo Treatment
Estimated effect:0.08112659407717893
New effect:-0.0012270350213641158
p value:0.44332637490518256


=== Causal Effect Per Factor ===
  weather_severity          causal effect = 0.0811
  nas_congestion            causal effect = 0.0968
  carrier_issues            causal effect = 0.0592
  late_aircraft             causal effect = 0.0175



Exports saved to outputs/
